# Distran: Amazon cross-media training and app content model
Run all on a CPU runtime. Downloads complete official 5-core ratings-only files, selects a deterministic 1/128 user sample, splits globally by time, fits BPR, compares popularity and validation-selected hybrids, and evaluates test once after selection. App content embeddings use the separate curated snapshot below. Research results are not production promotion.

Source: https://amazon-reviews-2023.github.io/data_processing/5core.html . Cite Hou et al., *Bridging Language and Items for Retrieval and Recommendation* (2024). 5-core eligibility was computed by the publisher on complete histories, so this is a retrospective benchmark. Raw ratings and hashed user maps stay in private artifacts, not notebook output. See TRAINING_RESULTS.md alongside this notebook.


In [ ]:
# @title Pin experiment dependencies
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","numpy==2.2.6","scikit-learn==1.6.1"])

0

In [ ]:
# @title Install versioned training code and curated catalog snapshot
from pathlib import Path
import json, sys
root=Path("/content/distran")
root.mkdir(parents=True,exist_ok=True)
files={'distran_ml/baseline.py': '"""Small, inspectable BPR experiment. CPU/Colab; not production serving."""\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\n\n\ndef temporal_split(events):\n    """Global time cutoffs prevent one user\'s future leaking into another\'s past."""\n    times = sorted({int(e[\'timestamp\']) for e in events})\n    if len(times) < 3:\n        raise ValueError(\'Need at least three distinct timestamps\')\n    ai = min(len(times)-3, max(0, int(len(times)*.6)-1))\n    bi = min(len(times)-2, max(ai+1, int(len(times)*.8)-1))\n    a, b = times[ai], times[bi]\n    return ([e for e in events if e[\'timestamp\'] <= a],\n            [e for e in events if a < e[\'timestamp\'] <= b],\n            [e for e in events if e[\'timestamp\'] > b])\n\n\ndef fit(events, dim=16, epochs=30, seed=42, lr=.03, reg=.01):\n    users = sorted({e[\'user_id\'] for e in events})\n    items = sorted({e[\'item_id\'] for e in events})\n    ui, ii = {x:i for i,x in enumerate(users)}, {x:i for i,x in enumerate(items)}\n    positives = defaultdict(set)\n    for e in events:\n        if e[\'rating\'] >= 4:\n            positives[ui[e[\'user_id\']]].add(ii[e[\'item_id\']])\n    rng = np.random.default_rng(seed)\n    U = rng.normal(0,.1,(len(users),dim)); V = rng.normal(0,.1,(len(items),dim))\n    # Complement precomputed for small experiments only. Replace at large scale.\n    # Keep complements as arrays: rng.choice(list) converts the entire list\n    # on every SGD step, making even bounded real-data runs unnecessarily slow.\n    negatives = {u: np.array(sorted(set(range(len(items)))-seen), dtype=np.int64)\n                 for u,seen in positives.items()}\n    pairs = [(u,i) for u,seen in positives.items() for i in sorted(seen) if len(negatives[u])]\n    if not pairs:\n        raise ValueError(\'No trainable positive/negative pairs\')\n    for _ in range(epochs):\n        for idx in rng.permutation(len(pairs)):\n            u,i = pairs[idx]; j = int(rng.choice(negatives[u]))\n            old = U[u].copy(); vi=V[i].copy(); vj=V[j].copy()\n            g = 1/(1+np.exp(np.clip(old @ (vi-vj), -30, 30)))\n            U[u] += lr*(g*(vi-vj)-reg*old)\n            V[i] += lr*(g*old-reg*vi); V[j] += lr*(-g*old-reg*vj)\n    return users, items, U, V\n\n\ndef evaluate(model, train, heldout, k=10):\n    users,items,U,V=model; ui={u:i for i,u in enumerate(users)}\n    seen=defaultdict(set); relevant=defaultdict(set)\n    for e in train: seen[e[\'user_id\']].add(e[\'item_id\'])\n    for e in heldout:\n        if e[\'rating\']>=4: relevant[e[\'user_id\']].add(e[\'item_id\'])\n    recalls=[]; ndcgs=[]; eligible=0\n    for u,truth in relevant.items():\n        if u not in ui: continue\n        truth=truth.intersection(items)-seen[u]\n        if not truth: continue\n        eligible+=1\n        ranked=sorted((i for i in range(len(items)) if items[i] not in seen[u]), key=lambda i: (-float(U[ui[u]]@V[i]),items[i]))[:k]\n        hits=[int(items[i] in truth) for i in ranked]\n        recalls.append(sum(hits)/len(truth))\n        dcg=sum(h/np.log2(r+2) for r,h in enumerate(hits))\n        ideal=sum(1/np.log2(r+2) for r in range(min(k,len(truth))))\n        ndcgs.append(dcg/ideal)\n    return {\'recall_at_k\': float(np.mean(recalls)) if recalls else None,\n            \'ndcg_at_k\': float(np.mean(ndcgs)) if ndcgs else None,\n            \'evaluated_users\':eligible,\'positive_heldout_users\':len(relevant), \'k\':k}\n\n\ndef demo():\n    events=[]\n    for t in range(1,31):\n        for u in range(12):\n            item=(u*3+t//2)%36\n            events.append(dict(user_id=f\'demo:{u}\',item_id=f\'{"game" if item%2 else "movie"}:{item}\',rating=5 if item%3==u%3 else 3,timestamp=t))\n    train,val,test=temporal_split(events)\n    model=fit(train)\n    return model, {\'validation\':evaluate(model,train,val),\'test\':evaluate(model,train,test),\'synthetic\':True}\n\nif __name__==\'__main__\':\n    model,metrics=demo(); print(json.dumps(metrics,indent=2))\n', 'experiment.py': '"""Complete-file Amazon 5-core experiment; run on CPU in Colab.\nRaw source IDs are used only in memory; exported factors contain salted hashes.\n"""\nimport csv, gzip, hashlib, json, os, platform, subprocess, sys, time, urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport numpy as np\nfrom distran_ml.baseline import temporal_split, fit\n\nBASE=\'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/\'\nSEED=42\n\ndef prepare(root):\n    salt=os.urandom(32)\n    events=[]; sources=[]; user_sets=[]\n    for domain,category in [(\'game\',\'Video_Games\'),(\'screen\',\'Movies_and_TV\')]:\n        path=root/(category+\'.csv.gz\'); url=BASE+path.name\n        print(\'Downloading complete file:\',category,flush=True)\n        if not path.exists():\n            with urllib.request.urlopen(url,timeout=180) as response, path.open(\'wb\') as out:\n                while chunk:=response.read(1024*1024):out.write(chunk)\n        digest=hashlib.sha256(path.read_bytes()).hexdigest()\n        users=set(); count=0; kept=0\n        with gzip.open(path,\'rt\') as stream:\n            for row in csv.DictReader(stream):\n                count+=1; raw=row[\'user_id\']; users.add(raw)\n                # Source-independent fixed hash => same user selected in both domains.\n                if int.from_bytes(hashlib.sha256(raw.encode()).digest()[:8],\'big\')%128:continue\n                rating=float(row[\'rating\']); timestamp=int(row[\'timestamp\'])\n                if not np.isfinite(rating) or not 1<=rating<=5:raise ValueError(\'Invalid rating\')\n                user=hashlib.sha256(salt+raw.encode()).hexdigest()\n                events.append(dict(user_id=user,item_id=domain+\':\'+row[\'parent_asin\'],rating=rating,timestamp=timestamp));kept+=1\n        user_sets.append(users)\n        sources.append(dict(url=url,sha256=digest,bytes=path.stat().st_size,rows=count,users=len(users),sampled_rows=kept))\n        print(category,sources[-1],flush=True)\n    overlap=len(user_sets[0]&user_sets[1]); del user_sets\n    unique={(e[\'user_id\'],e[\'item_id\'],e[\'timestamp\']):e for e in events}\n    return list(unique.values()),dict(sources=sources,full_file_overlap_users=overlap,duplicates_removed=len(events)-len(unique),sample=\'SHA256(source_user_id) first 8 bytes big endian modulo 128 == 0; complete file scans; no future-based user filtering\',pseudonyms=\'Random per-run salt, not exported; sampled cohort deterministic, pseudonyms differ each run\',limitations=\'Publisher 5-core selection uses complete history; global split is within that preselected corpus, not a prospective population study. screen combines movies and TV; no reliable separation in ratings-only IDs.\')\n\ndef ranking_metrics(scores, users, items, train, heldout, domain=None, source=None):\n    seen=defaultdict(set); relevant=defaultdict(set); positive_domains=defaultdict(set)\n    item_index={x:i for i,x in enumerate(items)}\n    for e in train:\n        seen[e[\'user_id\']].add(e[\'item_id\'])\n        if e[\'rating\']>=4:positive_domains[e[\'user_id\']].add(e[\'item_id\'].split(\':\')[0])\n    for e in heldout:\n        if e[\'rating\']>=4 and (domain is None or e[\'item_id\'].startswith(domain+\':\')):relevant[e[\'user_id\']].add(e[\'item_id\'])\n    rows=[]; recommended=set(); cold=repeat=0\n    for truth in relevant.values():cold+=sum(i not in item_index for i in truth)\n    for u,truth in relevant.items():repeat+=len(truth & seen[u])\n    for n,u in enumerate(users):\n        if source and source not in positive_domains[u]:continue\n        truth=(relevant[u]&item_index.keys())-seen[u]\n        if not truth:continue\n        candidates=np.array([i for i,x in enumerate(items) if x not in seen[u] and (domain is None or x.startswith(domain+\':\'))],dtype=int)\n        ranked=candidates[np.argsort(-scores[n,candidates],kind=\'stable\')[:10]]\n        recommended.update(items[i] for i in ranked)\n        hits=np.array([items[i] in truth for i in ranked],dtype=float)\n        recall=float(hits.sum()/len(truth)); ideal=sum(1/np.log2(r+2) for r in range(min(10,len(truth))))\n        ndcg=float(sum(h/np.log2(r+2) for r,h in enumerate(hits))/ideal)\n        rows.append((recall,ndcg))\n    a=np.array(rows); metrics=dict(evaluated_users=len(rows),positive_heldout_users=len(relevant),cold_item_positives=cold,repeat_positives=repeat,k=10,ranking=\'full train catalog, exclude all training interactions\',catalog_coverage=len(recommended)/max(1,sum(domain is None or i.startswith(domain+\':\') for i in items)))\n    for j,key in enumerate([\'recall_at_10\',\'ndcg_at_10\']):\n        metrics[key]=float(a[:,j].mean()) if len(rows) else None\n        if len(rows):\n            rng=np.random.default_rng(SEED); boots=[float(a[rng.integers(0,len(a),len(a)),j].mean()) for _ in range(400)]\n            metrics[key+\'_ci95\']=np.quantile(boots,[.025,.975]).tolist()\n    return metrics\n\ndef train_content(catalog, output):\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.decomposition import TruncatedSVD\n    docs=[\' \'.join([w[\'title\'],w[\'description\'],\' \'.join(w[\'tags\']),w[\'maker\']]) for w in catalog]\n    tf=TfidfVectorizer(stop_words=\'english\',ngram_range=(1,2),max_features=4096)\n    X=tf.fit_transform(docs)\n    svd=TruncatedSVD(n_components=min(8,len(catalog)-1),random_state=SEED)\n    vectors=svd.fit_transform(X);vectors/=np.maximum(np.linalg.norm(vectors,axis=1,keepdims=True),1e-12)\n    artifact=dict(schema=1,model_id=\'catalog-tfidf-svd-v1\',feature_schema=\'title-description-tags-maker-v1\',catalog_sha256=hashlib.sha256(json.dumps(catalog,sort_keys=True).encode()).hexdigest(),items=[w[\'id\'] for w in catalog],media=[w[\'medium\'] for w in catalog],vectors=vectors.tolist(),fallback=[w[\'id\'] for w in catalog],training=dict(seed=SEED,documents=len(docs),vocabulary=len(tf.vocabulary_),dimensions=vectors.shape[1],explained_variance=float(svd.explained_variance_ratio_.sum())),status=\'preview-content-only; no relevance-quality claim\')\n    (output/\'catalog-model.json\').write_text(json.dumps(artifact,indent=2))\n    # Safe JSON transformer export supports reproducing item embeddings without pickle.\n    (output/\'content-transform.json\').write_text(json.dumps(dict(vocabulary=tf.vocabulary_,idf=tf.idf_.tolist(),components=svd.components_.tolist()),default=int))\n    probes=[]\n    for n,w in enumerate(catalog):\n        order=np.argsort(-(vectors@vectors[n])); picks=[catalog[j][\'id\'] for j in order if j!=n][:4]\n        probes.append(dict(seed=w[\'id\'],recommendations=picks))\n    return dict(**artifact[\'training\'],examples=probes)\n\ndef run(root=Path(\'/content/distran\')):\n    started=time.time();output=root/\'artifacts\';output.mkdir(exist_ok=True,parents=True)\n    events,audit=prepare(root)\n    train,val,test=temporal_split(events)\n    print(\'Split\',len(train),len(val),len(test),flush=True)\n    users,items,U,V=fit(train,dim=16,epochs=30,seed=SEED)\n    print(\'BPR fit complete:\',len(users),\'users,\',len(items),\'items\',flush=True)\n    # Ranking by ordinal percentiles gives collaborative and popularity equal scale.\n    collab=U@V.T\n    counts=Counter(e[\'item_id\'] for e in train if e[\'rating\']>=4)\n    pop=np.array([counts[i] for i in items],dtype=float)\n    def percentile(a):\n        # Tie-preserving empirical CDF; equal counts receive identical score.\n        return np.searchsorted(np.sort(a,axis=-1),a,side=\'right\')/len(a)\n    pop=percentile(pop)\n    cf=np.stack([percentile(row) for row in collab])\n    popularity=np.broadcast_to(pop,cf.shape)\n    validation={}\n    for weight in [0,.25,.5,.75,1]:\n        validation[str(weight)]=ranking_metrics(weight*cf+(1-weight)*pop,users,items,train,val)\n        print(\'Validated weight\',weight,flush=True)\n    best=max(validation,key=lambda key: (validation[key][\'ndcg_at_10\'] or 0,-float(key)))\n    print(\'Validation selected CF weight\',best,flush=True)\n    models={\'popularity\':popularity,\'bpr\':cf,\'hybrid\':float(best)*cf+(1-float(best))*pop}\n    results={}\n    for name,scores in models.items():\n        results[name]={}\n        for cohort,domain,source in [(\'all\',None,None),(\'game\',\'game\',None),(\'screen\',\'screen\',None),(\'game_to_screen\',\'screen\',\'game\'),(\'screen_to_game\',\'game\',\'screen\')]:\n            results[name][cohort]=ranking_metrics(scores,users,items,train,test,domain,source)\n        print(\'Evaluated\',name,flush=True)\n    # Source-only fold-in probes: no target interactions in the user representation.\n    indices={x:i for i,x in enumerate(items)};profiles={}\n    for source,target in [(\'game\',\'screen\'),(\'screen\',\'game\')]:\n        P=np.zeros_like(U); n=Counter()\n        for e in train:\n            if e[\'rating\']>=4 and e[\'item_id\'].startswith(source+\':\'):\n                idx=users.index(e[\'user_id\']);P[idx]+=V[indices[e[\'item_id\']]];n[idx]+=1\n        for idx,count in n.items():P[idx]/=count\n        scores=P@V.T\n        profiles[source+\'_to_\'+target]=ranking_metrics(scores,users,items,train,test,target,source)\n    np.savez_compressed(output/\'bpr-factors.npz\',users=np.array(users),items=np.array(items),U=U,V=V)\n    content=train_content(json.loads((root/\'catalog.json\').read_text()),output)\n    report=dict(run_id=\'amazon-5core-20261116\',runtime=\'Google Colab CPU via colab CLI\',seed=SEED,environment=dict(python=sys.version,numpy=np.__version__,platform=platform.platform()),audit=audit,counts=dict(events=len(events),train=len(train),validation=len(val),test=len(test),users=len(users),items=len(items)),cutoffs=dict(train_max=max(e[\'timestamp\'] for e in train),validation_max=max(e[\'timestamp\'] for e in val),unit=\'Unix milliseconds\'),parameters=dict(dim=16,epochs=30,lr=.03,reg=.01,positive=\'rating >= 4\',negative=\'uniform unobserved-positive complement; unobserved is not known dislike\'),validation=validation,selected_cf_weight=float(best),test=results,source_only_fold_in=profiles,content=content,seconds=time.time()-started,promotion=\'Research only. Amazon IDs have no reviewed app catalog mapping; app uses separate content model. Warm cross-domain cohorts use all training interactions; source-only fold-in is reported separately. Social, graph, TV-only, and cold-item quality not evaluated.\')\n    (output/\'report.json\').write_text(json.dumps(report,indent=2))\n    (output/\'environment.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    print(json.dumps(report,indent=2),flush=True)\n    import zipfile\n    with zipfile.ZipFile(root/\'results.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for file in output.iterdir():z.write(file,file.name)\n    return report\n\nif __name__==\'__main__\':run()\n', 'catalog.json': '[\n  {\n    "id": "it-takes-two",\n    "title": "It Takes Two",\n    "medium": "game",\n    "description": "Two people, one extraordinary world. Play as Cody and May, a couple transformed into dolls, on a wonderfully unpredictable journey built entirely for two.",\n    "tags": [\n      "Co-op",\n      "Adventure",\n      "A little wonder"\n    ],\n    "maker": "Hazelight Studios"\n  },\n  {\n    "id": "cyberpunk",\n    "title": "Cyberpunk 2077",\n    "medium": "game",\n    "description": "Become V, a mercenary searching for a one-of-a-kind implant in Night City. A story about identity, impossible choices, and the people you find at the edge of the future.",\n    "tags": [\n      "Cyberpunk",\n      "Open world",\n      "Night city"\n    ],\n    "maker": "CD PROJEKT RED"\n  },\n  {\n    "id": "elden-ring",\n    "title": "Elden Ring",\n    "medium": "game",\n    "description": "Rise, Tarnished, and follow the grace of gold into the Lands Between. A vast, mysterious world where every ruin has a story and every distant light is an invitation.",\n    "tags": [\n      "Dark fantasy",\n      "Exploration",\n      "RPG"\n    ],\n    "maker": "FromSoftware"\n  },\n  {\n    "id": "disco-elysium",\n    "title": "Disco Elysium",\n    "medium": "game",\n    "description": "You are a detective with a city to unravel and a mind that will not stay quiet. Question the world, question yourself, and find something worth holding on to in Revachol.",\n    "tags": [\n      "Detective",\n      "Narrative",\n      "Weird cities"\n    ],\n    "maker": "ZA/UM"\n  },\n  {\n    "id": "baldurs-gate",\n    "title": "Baldur’s Gate 3",\n    "medium": "game",\n    "description": "Gather your party and return to the Forgotten Realms. An adventure shaped by friendship, freedom, and the choices that make a story yours.",\n    "tags": [\n      "RPG",\n      "Co-op",\n      "Fantasy"\n    ],\n    "maker": "Larian Studios"\n  },\n  {\n    "id": "outer-wilds",\n    "title": "Outer Wilds",\n    "medium": "game",\n    "description": "An open-world mystery about a solar system trapped in an endless time loop. Pack your curiosity, follow an unfamiliar signal, and discover what waits beyond the next horizon.",\n    "tags": [\n      "Exploration",\n      "Space",\n      "Mystery"\n    ],\n    "maker": "Mobius Digital"\n  },\n  {\n    "id": "exit-8",\n    "title": "The Exit 8",\n    "medium": "game",\n    "description": "You are trapped in an endless underground passageway. Observe your surroundings carefully. If you find an anomaly, turn back. Find the way to Exit 8.",\n    "tags": [\n      "Liminal",\n      "Horror",\n      "Short journey"\n    ],\n    "maker": "KOTAKE CREATE"\n  },\n  {\n    "id": "dune",\n    "title": "Dune: Part Two",\n    "medium": "movie",\n    "description": "Paul Atreides joins Chani and the Fremen while seeking revenge against the conspirators who destroyed his family. In the desert, a personal journey becomes something much larger.",\n    "tags": [\n      "Science fiction",\n      "Epic",\n      "World-building"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "blade-runner",\n    "title": "Blade Runner 2049",\n    "medium": "movie",\n    "description": "A new blade runner uncovers a secret that could change what remains of society. His search leads through a rain-soaked future, toward questions about memory and being human.",\n    "tags": [\n      "Cyberpunk",\n      "Atmospheric",\n      "Identity"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "arrival",\n    "title": "Arrival",\n    "medium": "movie",\n    "description": "When mysterious spacecraft appear across the world, a linguist searches for a way to communicate. A quiet, moving story about language, time, and the connections that shape a life.",\n    "tags": [\n      "Science fiction",\n      "Language",\n      "Contemplative"\n    ],\n    "maker": "Denis Villeneuve"\n  },\n  {\n    "id": "severance",\n    "title": "Severance",\n    "medium": "tv",\n    "description": "A team of office workers have surgically divided their memories between work and personal life. When the boundary begins to unravel, they discover that nothing about their work is ordinary.",\n    "tags": [\n      "Mystery",\n      "Identity",\n      "Uncanny"\n    ],\n    "maker": "Dan Erickson"\n  },\n  {\n    "id": "edgerunners",\n    "title": "Cyberpunk: Edgerunners",\n    "medium": "tv",\n    "description": "A street kid tries to survive in a city obsessed with technology and body modification. A standalone story in the world of Cyberpunk 2077, about found family and the price of a dream.",\n    "tags": [\n      "Anime",\n      "Cyberpunk",\n      "Found family"\n    ],\n    "maker": "Studio Trigger"\n  }\n]'}
for name,content in files.items():
    target=root/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(content)
sys.path.insert(0,str(root))

In [ ]:
# @title Preprocess, train, select on validation, and evaluate
from experiment import run
report=run(root)

Video_Games {'url': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/Video_Games.csv.gz', 'sha256': 'a2bde5f3b945960d161538c200dd87845e6ee471b46da96410dde61613c6901c', 'bytes': 12801967, 'rows': 814586, 'users': 94762, 'sampled_rows': 6676}


Movies_and_TV {'url': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/Movies_and_TV.csv.gz', 'sha256': '260d72b08d5e501276b349fed80ad23715dc56bd90fb89a52598ac7194a0ade5', 'bytes': 111047519, 'rows': 7441129, 'users': 657203, 'sampled_rows': 59323}


Split 39600 13204 13195


BPR fit complete: 4718 users, 24390 items


Validated weight 0


Validated weight 0.25


... (4 outputs left out)

Evaluated popularity


Evaluated bpr


Evaluated hybrid


{
  "run_id": "amazon-5core-20261116",
  "runtime": "Google Colab CPU via colab CLI",
  "seed": 42,
  "environment": {
    "python": "3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]",
    "numpy": "2.2.6",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39"
  },
  "audit": {
    "sources": [
      {
        "url": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/Video_Games.csv.gz",
        "sha256": "a2bde5f3b945960d161538c200dd87845e6ee471b46da96410dde61613c6901c",
        "bytes": 12801967,
        "rows": 814586,
        "users": 94762,
        "sampled_rows": 6676
      },
      {
        "url": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/5core/rating_only/Movies_and_TV.csv.gz",
        "sha256": "260d72b08d5e501276b349fed80ad23715dc56bd90fb89a52598ac7194a0ade5",
        "bytes": 111047519,
        "rows": 7441129,
        "users": 657203,
        "sampled_rows": 59323
      }
    ],
    "full_file_overlap_

In [ ]:
# @title Inspect serving model examples
print(json.dumps(report["content"],indent=2))
print("Artifacts:",root/"results.zip")

{
  "seed": 42,
  "documents": 12,
  "vocabulary": 433,
  "dimensions": 8,
  "explained_variance": 0.6711771554759998,
  "examples": [
    {
      "seed": "it-takes-two",
      "recommendations": [
        "baldurs-gate",
        "exit-8",
        "dune",
        "cyberpunk"
      ]
    },
    {
      "seed": "cyberpunk",
      "recommendations": [
        "edgerunners",
        "outer-wilds",
        "blade-runner",
        "disco-elysium"
      ]
    },
    {
      "seed": "elden-ring",
      "recommendations": [
        "baldurs-gate",
        "outer-wilds",
        "arrival",
        "cyberpunk"
      ]
    },
    {
      "seed": "disco-elysium",
      "recommendations": [
        "arrival",
        "cyberpunk",
        "severance",
        "it-takes-two"
      ]
    },
    {
      "seed": "baldurs-gate",
      "recommendations": [
        "it-takes-two",
        "elden-ring",
        "cyberpunk",
        "blade-runner"
      ]
    },
    {
      "seed": "outer-wilds",
      "recom